In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif, VarianceThreshold

INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Cantidad de features a seleccionar para métodos SelectKBest
K = 100

resultados_resumen = []

for norm_type in os.listdir(INPUT_PATH):
    full_path = os.path.join(INPUT_PATH, norm_type)
    if not os.path.isdir(full_path): continue

    for version in ['ORIGINAL', 'FE']:
        base_name = f"{norm_type}_{version}"
        print(f"\n🔍 Procesando {base_name}...")

        try:
            X_train = pd.read_parquet(os.path.join(full_path, f"X_train_{base_name}.parquet"))
            X_test = pd.read_parquet(os.path.join(full_path, f"X_test_{base_name}.parquet"))
            y_train = pd.read_parquet(os.path.join(full_path, f"y_train_{base_name}.parquet"))
            y_test = pd.read_parquet(os.path.join(full_path, f"y_test_{base_name}.parquet"))
        except Exception as e:
            print(f"❌ Error cargando datos para {base_name}: {e}")
            continue

        # Asegurar mismas columnas en test que en train
        X_test = X_test[X_train.columns.intersection(X_test.columns)]

        techniques = {}

        if (X_train >= 0).all().all():
            techniques["CHI2"] = SelectKBest(score_func=chi2, k=K)
        else:
            print(f"⚠️ Saltando CHI2 por contener negativos en {base_name}")

        techniques["ANOVA"] = SelectKBest(score_func=f_classif, k=K)
        techniques["MI"] = SelectKBest(score_func=mutual_info_classif, k=K)
        techniques["VAR"] = VarianceThreshold()
        techniques["ALL"] = "ALL"

        for name, selector in techniques.items():
            try:
                if name == "ALL":
                    cols = X_train.columns
                    X_train_sel = X_train.copy()
                    X_test_sel = X_test[cols].copy()
                elif name == "VAR":
                    selector.fit(X_train)
                    mask = selector.get_support()
                    cols = X_train.columns[mask]
                    X_train_sel = X_train[cols]
                    X_test_sel = X_test[cols]
                else:
                    selector.fit(X_train, y_train.values.ravel())
                    mask = selector.get_support()
                    cols = X_train.columns[mask]
                    X_train_sel = X_train[cols]
                    X_test_sel = X_test[cols]

                    # Guardar métricas
                    scores = selector.scores_
                    df_scores = pd.DataFrame({
                        "feature": X_train.columns,
                        "score": scores
                    })
                    df_scores.sort_values("score", ascending=False, inplace=True)
                    df_scores.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{base_name}_{name}.csv"), index=False)
                    print(f"📅 Métricas guardadas: metricas_{base_name}_{name}.csv")

                # Guardar datasets
                prefix = f"{base_name}_{name}"
                X_train_sel.to_parquet(os.path.join(OUTPUT_PATH, f"X_train_{prefix}.parquet"))
                X_test_sel.to_parquet(os.path.join(OUTPUT_PATH, f"X_test_{prefix}.parquet"))
                y_train.to_parquet(os.path.join(OUTPUT_PATH, f"y_train_{prefix}.parquet"))
                y_test.to_parquet(os.path.join(OUTPUT_PATH, f"y_test_{prefix}.parquet"))

                print(f"✔ {base_name} - {name}: {len(cols)} variables seleccionadas")
                resultados_resumen.append({
                    "combinacion": base_name,
                    "tecnica": name,
                    "columnas_seleccionadas": len(cols)
                })

            except Exception as e:
                print(f"❌ Error al guardar {name} en {base_name}: {e}")

# Guardar resumen final
df_resumen = pd.DataFrame(resultados_resumen)
df_resumen.to_csv(os.path.join(OUTPUT_PATH, "resumen_cantidad_variables_seleccionadas.csv"), index=False)
print("\n📊 Resumen guardado en: resumen_cantidad_variables_seleccionadas.csv")



🔍 Procesando MaxAbs_ORIGINAL...
⚠️ Saltando CHI2 por contener negativos en MaxAbs_ORIGINAL
📅 Métricas guardadas: metricas_MaxAbs_ORIGINAL_ANOVA.csv
✔ MaxAbs_ORIGINAL - ANOVA: 100 variables seleccionadas
📅 Métricas guardadas: metricas_MaxAbs_ORIGINAL_MI.csv
✔ MaxAbs_ORIGINAL - MI: 100 variables seleccionadas
✔ MaxAbs_ORIGINAL - VAR: 539 variables seleccionadas
✔ MaxAbs_ORIGINAL - ALL: 539 variables seleccionadas

🔍 Procesando MaxAbs_FE...
⚠️ Saltando CHI2 por contener negativos en MaxAbs_FE
📅 Métricas guardadas: metricas_MaxAbs_FE_ANOVA.csv
✔ MaxAbs_FE - ANOVA: 100 variables seleccionadas


KeyboardInterrupt: 